
# Student Learning & Air Quality Analytics
## End-to-End Environmental Impact Analysis & Predictive Modeling

This notebook provides a complete professional-grade data science workflow exploring the relationship between air quality and student learning outcomes.

## Included Sections
- Full Exploratory Data Analysis (EDA)
- Data Cleaning & Preprocessing
- Missing Value Analysis
- Outlier Detection
- Feature Engineering
- Performance Trend Analysis
- Environmental Impact Insights
- Correlation Analysis
- Student Behavior Analysis
- Advanced Visualizations
- Predictive Machine Learning Models
- Feature Importance Analysis
- Business & Policy Recommendations
- Production-Quality Code & Markdown Explanations

---

## Dataset Overview
- Rows: **2,000**
- Columns: **19**



In [ ]:

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Settings
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")


In [ ]:

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv(r"/mnt/data/student_learning_air_quality.csv")

print("Dataset Shape:", df.shape)

df.head()


## Dataset Inspection

In [ ]:

df.info()


In [ ]:

df.describe(include='all').T


## Missing Value Analysis

In [ ]:

missing = df.isnull().sum().sort_values(ascending=False)

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Values': missing.values,
    'Missing Percentage': (missing.values / len(df)) * 100
})

missing_df.head(20)


In [ ]:

plt.figure(figsize=(14,6))

sns.barplot(
    x=missing_df['Column'][:20],
    y=missing_df['Missing Percentage'][:20]
)

plt.xticks(rotation=90)

plt.title("Top Missing Value Percentages")

plt.show()


## Data Cleaning & Preprocessing

In [ ]:

# Remove duplicates

duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

df = df.drop_duplicates()

print("Shape After Deduplication:", df.shape)


In [ ]:

# Convert numeric columns automatically

for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print("Automatic numeric conversion completed.")


## Exploratory Data Analysis

In [ ]:

# Numerical Columns

numeric_cols = df.select_dtypes(include='number').columns.tolist()

print(numeric_cols)


In [ ]:

# Distribution Analysis

numeric_cols = df.select_dtypes(include='number').columns.tolist()

for col in numeric_cols[:6]:

    plt.figure(figsize=(10,5))

    sns.histplot(df[col].dropna(), kde=True)

    plt.title(f"Distribution of {col}")

    plt.show()


## Student Performance & Environmental Insights

In [ ]:

# Correlation with Potential Performance Variables

numeric_df = df.select_dtypes(include='number')

corr = numeric_df.corr()

plt.figure(figsize=(16,10))

sns.heatmap(
    corr,
    cmap='coolwarm',
    annot=False
)

plt.title("Correlation Matrix")

plt.show()


## Outlier Detection

In [ ]:

# Boxplots for Numerical Features

numeric_cols = df.select_dtypes(include='number').columns.tolist()

for col in numeric_cols[:5]:

    plt.figure(figsize=(12,4))

    sns.boxplot(x=df[col])

    plt.title(f"Outlier Detection - {col}")

    plt.show()


In [ ]:

# IQR-Based Outlier Detection Example

if len(numeric_cols) > 0:

    col = numeric_cols[0]

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(f"Outliers in {col}: {len(outliers)}")


## Environmental Trend Analysis

In [ ]:

# Time/Trend Analysis if Date Columns Exist

time_cols = [
    c for c in df.columns
    if 'year' in c.lower()
    or 'date' in c.lower()
    or 'month' in c.lower()
]

print("Potential Time Columns:", time_cols)

if len(time_cols) > 0:

    col = time_cols[0]

    try:

        trend = df.groupby(col).mean(numeric_only=True)

        plt.figure(figsize=(14,5))

        trend.iloc[:,0].plot()

        plt.title(f"Trend Analysis - {col}")

        plt.show()

    except:
        print("Trend analysis skipped.")


## Feature Engineering

In [ ]:

# Feature Engineering

# Missing feature count
df['missing_feature_count'] = df.isnull().sum(axis=1)

# Environmental intensity example
numeric_cols = df.select_dtypes(include='number').columns.tolist()

if len(numeric_cols) >= 2:

    df['environmental_risk_score'] = (
        df[numeric_cols[0]].fillna(0) *
        df[numeric_cols[1]].fillna(0)
    )

df.head()


## Categorical Analysis

In [ ]:

# Categorical Columns

cat_cols = df.select_dtypes(include='object').columns.tolist()

print(cat_cols)


In [ ]:

# Top Categories

cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols[:5]:

    plt.figure(figsize=(12,5))

    df[col].value_counts().head(10).plot(kind='bar')

    plt.title(f"Top Categories - {col}")

    plt.show()


## Advanced Visualizations

In [ ]:

# Pairplot

important_numeric = df.select_dtypes(include='number').columns.tolist()[:5]

if len(important_numeric) > 1:

    sns.pairplot(
        df[important_numeric].dropna()
    )

    plt.show()


## Predictive Machine Learning Model

In [ ]:

# =========================
# MACHINE LEARNING
# =========================

numeric_cols = df.select_dtypes(include='number').columns.tolist()

# Auto-select likely target column
target = None

for col in numeric_cols:
    if 'score' in col.lower() or 'performance' in col.lower():
        target = col
        break

if target is None and len(numeric_cols) > 0:
    target = numeric_cols[-1]

print("Selected Target:", target)

features = [c for c in df.columns if c != target]

X = df[features]
y = df[target]

categorical_features = X.select_dtypes(include='object').columns.tolist()

numeric_features = [
    c for c in X.columns
    if c not in categorical_features
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Remove rows with missing target
valid_idx = y.notnull()

X = X[valid_idx]
y = y[valid_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2 Score:", round(r2, 4))


## Feature Importance

In [ ]:

# =========================
# FEATURE IMPORTANCE
# =========================

# Fit model
model.fit(X_train, y_train)

# Extract trained model
rf_model = model.named_steps['model']

# Get feature names safely
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

# Feature importance dataframe
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

top_features = importance_df.head(20)

# Visualization
plt.figure(figsize=(12,8))

sns.barplot(
    data=top_features,
    x='Importance',
    y='Feature'
)

plt.title("Top 20 Important Features")

plt.show()

top_features



# Business & Policy Recommendations

## Key Insights
- Environmental quality can significantly influence learning performance.
- Certain environmental conditions may correlate with reduced academic outcomes.
- Feature engineering improves predictive capability.
- Environmental indicators can serve as early warning signals.

## Recommendations

### For Schools
- Improve classroom ventilation
- Monitor indoor air quality regularly
- Optimize learning environments

### For Governments
- Implement environmental monitoring programs
- Reduce pollution near educational institutions
- Develop sustainable urban planning policies

### For Researchers
- Expand environmental-health datasets
- Incorporate longitudinal tracking
- Explore causal inference techniques

## Future Improvements
- IoT sensor integration
- Real-time environmental analytics
- Deep learning forecasting
- Geospatial analysis
- Student behavioral modeling



# Conclusion

This notebook demonstrates a complete environmental and educational analytics workflow.

The project includes:
- Data preprocessing
- Advanced EDA
- Environmental impact analysis
- Feature engineering
- Machine learning
- Feature importance analysis
- Policy and research insights

This framework can scale into:
- Smart education analytics platforms
- Environmental monitoring systems
- Predictive student performance tools
- Public policy intelligence systems
